# Optimization Project — Final Notebook
**Parts:**

1. **Ideal Case (Part 1)** — Budgeting problem (no site selection, no distance constraints, no segmented costs)
2. **Realistic Case (Part 2)** — Piecewise expansion costs, candidate sites with sizes, and distance constraints

**Data folder (relative):** `./ChildCareDeserts_Data`

> Working directory expected by Oscar: `/Users/oscar/Documents/OptiProject`


In [13]:

# === Imports & Config ===
import os
import math
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from sklearn.neighbors import BallTree

# Ensure prints have full width
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 50)

# Paths
DATA_DIR = "./ChildCareDeserts_Data"
print("CWD:", os.getcwd())
print("Using data dir:", os.path.abspath(DATA_DIR))

def resolve_col(df, candidates, must=True, ctx=None):
    # normalize: strip spaces, lowercase for matching
    col_map = {c.strip().lower().replace(" ", ""): c for c in df.columns}
    for cand in candidates:
        key = cand.strip().lower().replace(" ", "")
        if key in col_map:
            return col_map[key]
    if must:
        raise KeyError(
            f"Could not resolve column for {ctx or candidates} "
            f"in df with columns: {list(df.columns)}"
        )
    return None


# Helper: check presence of all files
files = [
    "avg_individual_income.csv",
    "employment_rate.csv",
    "population.csv",
    "child_care_regulated.csv",
    "potential_locations.csv"
]
for f in files:
    path = os.path.join(DATA_DIR, f)
    print(f"{f}: {'Found' if os.path.exists(path) else 'Missing'}")


CWD: /Users/oscar/Documents/CODE/OptiProject
Using data dir: /Users/oscar/Documents/CODE/OptiProject/ChildCareDeserts_Data
avg_individual_income.csv: Found
employment_rate.csv: Found
population.csv: Found
child_care_regulated.csv: Found
potential_locations.csv: Found


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

## Load data

In [15]:

# === Load CSVs ===
inc = pd.read_csv(os.path.join(DATA_DIR, "avg_individual_income.csv"))
emp = pd.read_csv(os.path.join(DATA_DIR, "employment_rate.csv"))
pop = pd.read_csv(os.path.join(DATA_DIR, "population.csv"))
fac = pd.read_csv(os.path.join(DATA_DIR, "child_care_regulated.csv"))
cand = pd.read_csv(os.path.join(DATA_DIR, "potential_locations.csv"))

print("Loaded shapes:", {k:v.shape for k,v in {'income':inc,'employment':emp,'population':pop,'facilities':fac,'candidates':cand}.items()})


Loaded shapes: {'income': (1534, 2), 'employment': (1375, 2), 'population': (1646, 20), 'facilities': (15604, 15), 'candidates': (215400, 3)}


In [16]:
# Compute derived population columns from raw age buckets
pop['-5'] = pd.to_numeric(pop['-5'], errors='coerce').fillna(0)
pop['5-9'] = pd.to_numeric(pop['5-9'], errors='coerce').fillna(0)
pop['10-14'] = pd.to_numeric(pop['10-14'], errors='coerce').fillna(0)

pop['pop_0_5'] = pop['-5']
pop['pop_5_12'] = pop['5-9'] + 0.5 * pop['10-14']  # half of 10-14 = ages 10-12
pop['pop_total_0_12'] = pop['pop_0_5'] + pop['pop_5_12']

# Rename income and employment columns
inc = inc.rename(columns={'average income': 'average_income'})
emp = emp.rename(columns={'employment rate': 'employment_rate'})

## Normalize / resolve key columns

In [ ]:

# === Resolve ZIP columns ===
zip_inc = resolve_col(inc, ["zipcode","zip","ZIP","Zipcode","postal_code"], True, "income.zip")
zip_emp = resolve_col(emp, ["zipcode","zip","ZIP","Zipcode","postal_code"], True, "employment.zip")
zip_pop = resolve_col(pop, ["zipcode","zip","ZIP","Zipcode","postal_code"], True, "population.zip")

# Resolve metrics
col_income = resolve_col(inc, ["avg_individual_income","average income","income","avg_income"], True, "income.value")
col_emp    = resolve_col(emp, ["employment_rate","employment rate","employment","emp_rate"], True, "employment.rate")
col_p012   = resolve_col(pop, ["pop_total_0_12","population_0_12","pop_0_12","pop_total"], True, "population 0-12")
col_p05    = resolve_col(pop, ["pop_0_5","population_0_5","pop05"], True, "population 0-5")

# Prepare master by ZIP
master = (inc[[zip_inc, col_income]].rename(columns={zip_inc:"zipcode", col_income:"avg_individual_income"})
          .merge(emp[[zip_emp, col_emp]].rename(columns={zip_emp:"zipcode", col_emp:"employment_rate"}), on="zipcode", how="outer")
          .merge(pop[[zip_pop, col_p012, col_p05]].rename(columns={zip_pop:"zipcode", col_p012:"pop_total_0_12", col_p05:"pop_0_5"}), on="zipcode", how="outer"))

# Clean NaNs
for c in ["avg_individual_income", "employment_rate", "pop_total_0_12", "pop_0_5"]:
    master[c] = master[c].fillna(0.0)

print(master.head())
print("Master rows:", len(master))


## Facilities (existing) and capacity by ZIP

In [ ]:

# Resolve facility columns
f_zip = resolve_col(fac, ["zipcode","zip","ZIP","Zipcode","postal_code"], True, "facilities.zip")
f_id  = resolve_col(fac, ["facility_id","id","facility","name"], True, "facilities.id")
f_lat = resolve_col(fac, ["lat","latitude","Latitude"], True, "facilities.lat")
f_lon = resolve_col(fac, ["lon","lng","longitude","Longitude"], True, "facilities.lon")
f_cap = resolve_col(fac, ["capacity","cap","n","n_f","current_capacity"], True, "facilities.capacity")
f_cap05 = resolve_col(fac, ["cap_0_5","capacity_0_5","n_0_5"], False, "facilities.cap_0_5")

fac2 = fac.rename(columns={f_zip:"zipcode", f_id:"facility_id", f_lat:"lat", f_lon:"lon", f_cap:"capacity"})
if f_cap05 and f_cap05 in fac.columns:
    fac2["cap_0_5"] = fac[f_cap05].fillna(0.0)
else:
    # If not provided, assume proportion 0.25 of total as a fallback (will be over-written by constraints anyway)
    fac2["cap_0_5"] = 0.25 * fac2["capacity"]

fac2["capacity"] = fac2["capacity"].fillna(0.0)
fac2["cap_0_5"] = fac2["cap_0_5"].fillna(0.0)

# Existing totals by ZIP
by_zip = fac2.groupby("zipcode").agg(cap_total_zip=("capacity","sum"),
                                     cap_0_5_zip=("cap_0_5","sum")).reset_index()

master = master.merge(by_zip, on="zipcode", how="left")
master["cap_total_zip"] = master["cap_total_zip"].fillna(0.0)
master["cap_0_5_zip"]   = master["cap_0_5_zip"].fillna(0.0)

print("Facilities sample:"); print(fac2.head())
print("ZIP aggregates sample:"); print(master.head())


## Candidate sites & size dictionary

In [ ]:

# Resolve candidate columns
c_zip = resolve_col(cand, ["zipcode","zip","ZIP","Zipcode","postal_code"], True, "candidates.zip")
c_id  = resolve_col(cand, ["site_id","id","site","name"], True, "candidates.id")
c_lat = resolve_col(cand, ["lat","latitude","Latitude"], True, "candidates.lat")
c_lon = resolve_col(cand, ["lon","lng","longitude","Longitude"], True, "candidates.lon")

cand2 = cand.rename(columns={c_zip:"zipcode", c_id:"site_id", c_lat:"lat", c_lon:"lon"})

# Size dictionary (fixed by spec)
sizes = {'S': {'tot':100, 'max05':50,  'cost':65000},
         'M': {'tot':200, 'max05':100, 'cost':95000},
         'L': {'tot':400, 'max05':200, 'cost':115000}}

# Facility params with expansion segment caps
facility_params = {}
for _, r in fac2.iterrows():
    n_f = float(r["capacity"])
    facility_params[r["facility_id"]] = {
        "zipcode": r["zipcode"],
        "lat": r["lat"],
        "lon": r["lon"],
        "n_f": n_f,
        "s1_max": 0.10 * n_f,
        "s2_max": 0.05 * n_f,
        "s3_max": 0.05 * n_f,
        "overall_cap": 0.20 * n_f,
    }

# Site → ZIP map
site_zip_map = {r["site_id"]: r["zipcode"] for _, r in cand2.iterrows()}

print("facility_params sample:", list(facility_params.items())[:2])
print("site_zip_map sample:", list(site_zip_map.items())[:3])


## High-demand classification (OR rule: emp ≥ 0.6 OR income ≤ 60000)

In [ ]:

master["is_high_demand"] = (
    (master["employment_rate"] >= 0.6) | (master["avg_individual_income"] <= 60000)
).astype(int)

print(master[["zipcode","employment_rate","avg_individual_income","is_high_demand"]].head())


## Distance conflicts (new–new and new–old) with threshold 0.06 miles

In [ ]:

# Build BallTrees in radians for haversine
def to_radians(df):
    return np.radians(df[["lat","lon"]].values)

def miles_to_radians(miles):
    EARTH_RADIUS_MILES = 3958.7613
    return miles / EARTH_RADIUS_MILES

thr_rad = miles_to_radians(0.06)

# New–new: candidate sites within 0.06 miles
cand_coords = to_radians(cand2)
tree_cand = BallTree(cand_coords, metric='haversine')
pairs_new_new = []
for i, coord in enumerate(cand_coords):
    idxs = tree_cand.query_radius([coord], r=thr_rad)[0]
    for j in idxs:
        if j > i:  # avoid duplicates and self
            s1 = cand2.iloc[i]["site_id"]
            s2 = cand2.iloc[j]["site_id"]
            # same ZIP only (per spec aggregation)
            if cand2.iloc[i]["zipcode"] == cand2.iloc[j]["zipcode"]:
                pairs_new_new.append((cand2.iloc[i]["zipcode"], s1, s2))

from collections import defaultdict
new_new_conflicts = defaultdict(list)
for z, s1, s2 in pairs_new_new:
    new_new_conflicts[z].append((s1, s2))

# New–old: site vs existing facility within 0.06 miles
fac_coords = to_radians(fac2)
tree_fac = BallTree(fac_coords, metric='haversine')
pairs_new_old = []
for i, coord in enumerate(cand_coords):
    idxs = tree_fac.query_radius([coord], r=thr_rad)[0]
    for j in idxs:
        s = cand2.iloc[i]["site_id"]
        f = fac2.iloc[j]["facility_id"]
        if cand2.iloc[i]["zipcode"] == fac2.iloc[j]["zipcode"]:
            pairs_new_old.append((cand2.iloc[i]["zipcode"], s, f))

new_old_conflicts = defaultdict(list)
for z, s, f in pairs_new_old:
    new_old_conflicts[z].append((s, f))

print("new_new_conflicts sample:", list(new_new_conflicts.items())[:2])
print("new_old_conflicts sample:", list(new_old_conflicts.items())[:2])


# PART 1 — Ideal Case (Budgeting Problem)

In [ ]:

# Model — Ideal Case
m_ideal = gp.Model("ideal_case")

# Variables (per existing facility)
x_f = {f: m_ideal.addVar(lb=0, ub=facility_params[f]['overall_cap'], name=f"x_{f}")
       for f in facility_params}
new05_f = {f: m_ideal.addVar(lb=0, ub=facility_params[f]['overall_cap'], name=f"new05_{f}")
           for f in facility_params}

# Objective
expansion_cost_ideal = gp.quicksum(20000 + 200 * x_f[f] for f in facility_params)
equipment_cost_ideal = 100 * gp.quicksum(new05_f[f] for f in facility_params)
m_ideal.setObjective(expansion_cost_ideal + equipment_cost_ideal, GRB.MINIMIZE)

# Coverage constraints by ZIP
for _, row in master.iterrows():
    z = row["zipcode"]
    threshold = 0.5 if row["is_high_demand"] == 1 else (1/3)
    existing_total_zip = row["cap_total_zip"]
    existing_05_zip = row["cap_0_5_zip"]

    expansion_total_zip = gp.quicksum(x_f[f] for f, p in facility_params.items() if p["zipcode"] == z)
    new05_total_zip = gp.quicksum(new05_f[f] for f, p in facility_params.items() if p["zipcode"] == z)

    m_ideal.addConstr(existing_total_zip + expansion_total_zip >= threshold * row["pop_total_0_12"],
                      name=f"cov_total_{z}")
    m_ideal.addConstr(existing_05_zip + new05_total_zip >= (2/3) * row["pop_0_5"],
                      name=f"cov_05_{z}")

m_ideal.optimize()

if m_ideal.status == GRB.OPTIMAL:
    print("Ideal — Total cost:", f"${m_ideal.ObjVal:,.2f}")
else:
    print("Ideal — Optimization status:", m_ideal.status)


# PART 2 — Realistic Case (Piecewise + Sites + Distance)

In [ ]:

# Model — Realistic Case
m = gp.Model("realistic_case")

# Decision variables — existing facilities (piecewise)
x_f1 = {}; x_f2 = {}; x_f3 = {}; x_f_0_5 = {}
for f, params in facility_params.items():
    x_f1[f]    = m.addVar(lb=0, ub=params['s1_max'], name=f"x_f1_{f}")
    x_f2[f]    = m.addVar(lb=0, ub=params['s2_max'], name=f"x_f2_{f}")
    x_f3[f]    = m.addVar(lb=0, ub=params['s3_max'], name=f"x_f3_{f}")
    x_f_0_5[f] = m.addVar(lb=0, ub=params['overall_cap'], name=f"x_f_0_5_{f}")

# Candidate site decision variables
y = {}; new05 = {}
for s, row in cand2.set_index("site_id").iterrows():
    y[s] = {k: m.addVar(vtype=GRB.BINARY, name=f"y_{s}_{k}") for k in sizes}
    new05[s] = m.addVar(lb=0, ub=max(sizes[k]['max05'] for k in sizes), name=f"new05_{s}")

# Expansion costs (unit) — include +20000 / n_f
c1 = {f: 200  + 20000 / facility_params[f]['n_f'] for f in facility_params}
c2 = {f: 400  + 20000 / facility_params[f]['n_f'] for f in facility_params}
c3 = {f: 1000 + 20000 / facility_params[f]['n_f'] for f in facility_params}

# Objective
expansion_cost = gp.quicksum(c1[f]*x_f1[f] + c2[f]*x_f2[f] + c3[f]*x_f3[f] for f in facility_params)
construction_cost = gp.quicksum(sizes[k]['cost'] * y[s][k] for s in y for k in y[s])
equipment_cost = 100 * (gp.quicksum(x_f_0_5[f] for f in x_f_0_5) + gp.quicksum(new05[s] for s in new05))
m.setObjective(expansion_cost + construction_cost + equipment_cost, GRB.MINIMIZE)

# Constraints — segment caps and 0–5 linkage
for f, params in facility_params.items():
    m.addConstr(x_f1[f] + x_f2[f] + x_f3[f] <= params['overall_cap'], name=f"cap20_{f}")
    m.addConstr(x_f_0_5[f] <= x_f1[f] + x_f2[f] + x_f3[f], name=f"x05_leq_total_{f}")

# Site size selection and new05 bounds
for s in y:
    m.addConstr(gp.quicksum(y[s][k] for k in y[s]) <= 1, name=f"site_select_{s}")
    m.addConstr(new05[s] <= gp.quicksum(sizes[k]['max05'] * y[s][k] for k in y[s]), name=f"new05_max_{s}")

# Coverage constraints by ZIP
for _, row in master.iterrows():
    z = row["zipcode"]
    threshold = 0.5 if row["is_high_demand"] == 1 else (1/3)

    existing_total_zip = row["cap_total_zip"]
    existing_05_zip = row["cap_0_5_zip"]

    expansion_total_zip = gp.quicksum(
        (x_f1[f] + x_f2[f] + x_f3[f]) for f, p in facility_params.items() if p["zipcode"] == z
    )
    expansion_05_zip = gp.quicksum(x_f_0_5[f] for f, p in facility_params.items() if p["zipcode"] == z)

    newtotal_zip = gp.quicksum(
        gp.quicksum(sizes[k]["tot"] * y[s][k] for k in y[s]) for s in y if site_zip_map.get(s) == z
    )
    new05_zip = gp.quicksum(new05[s] for s in new05 if site_zip_map.get(s) == z)

    m.addConstr(existing_total_zip + expansion_total_zip + newtotal_zip >= threshold * row["pop_total_0_12"],
                name=f"cov_total_{z}")
    m.addConstr(existing_05_zip + expansion_05_zip + new05_zip >= (2/3) * row["pop_0_5"],
                name=f"cov_05_{z}")

# Distance constraints — new–new and new–old
for z, pairs in new_new_conflicts.items():
    for s1, s2 in pairs:
        m.addConstr(gp.quicksum(y[s1][k] for k in y[s1]) + gp.quicksum(y[s2][k] for k in y[s2]) <= 1,
                    name=f"conflict_newnew_{z}_{s1}_{s2}")

for z, pairs in new_old_conflicts.items():
    for s, f in pairs:
        # forbid new site s if too close to existing f
        m.addConstr(gp.quicksum(y[s][k] for k in y[s]) <= 0, name=f"conflict_newold_{z}_{s}_{f}")

# Solve
m.optimize()

# Debug: cost components
if m.status == GRB.OPTIMAL:
    exp_val = sum((c1[f]*x_f1[f].X + c2[f]*x_f2[f].X + c3[f]*x_f3[f].X) for f in facility_params)
    con_val = sum(sizes[k]['cost'] * y[s][k].X for s in y for k in y[s])
    eqp_val = 100 * (sum(x_f_0_5[f].X for f in x_f_0_5) + sum(new05[s].X for s in new05))
    print(f"Expansion cost:   ${exp_val:,.2f}")
    print(f"Construction cost:${con_val:,.2f}")
    print(f"Equipment cost:   ${eqp_val:,.2f}")
    print(f"Total cost:       ${m.ObjVal:,.2f}")
else:
    print("Realistic — Optimization status:", m.status)
